# 03 — TCP Models (OS proxy + RANO v3)

- Pooled TCP curves (OS ≥ median)
- RANO non-PD vs OS on same patients
- Within-arm DVH → RANO (60 / 40 Gy)
- Cox OS ~ EQD2 + RANO


In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.config import DATA_PROCESSED, FIGURES_DIR, REPORTS_DIR
from src.models.poisson_tcp import PoissonTCPModel
from src.models.logistic_tcp import LogisticTCPModel
from src.models.probit_tcp import ProbitTCPModel
from src.models.model_comparison import run_model_comparison
from src.analysis.rano_tcp_comparison import run_rano_tcp_comparison
from src.analysis.within_arm_rano_tcp import run_within_arm_rano_analysis


In [2]:
modeling = pd.read_csv(DATA_PROCESSED / "modeling_table.csv")
median_os = modeling["survival_weeks"].median()
outcomes = (modeling["survival_weeks"] >= median_os).astype(float).to_numpy()
doses = modeling["eqd2_gy"].to_numpy()
print(f"n={len(modeling)}, OS median={median_os:.0f} wk, RANO labels={modeling['rano_controlled_t1'].notna().sum()}")


n=190, OS median=51 wk, RANO labels=137


In [3]:
models = [
    ("Poisson", PoissonTCPModel(d50_init=55, gamma50_init=1.5)),
    ("Logistic", LogisticTCPModel(d50_init=53, k_init=0.1)),
    ("Probit", ProbitTCPModel(d50_init=53, sigma_init=10)),
]
d_grid = np.linspace(35, 65, 200)
fig, ax = plt.subplots(figsize=(7, 4.5))
for name, m in models:
    m.fit(doses, outcomes)
    ax.plot(d_grid, m.predict(d_grid), label=name, lw=2)
ax.set_xlabel("EQD2 (Gy)")
ax.set_ylabel("TCP (OS ≥ median proxy)")
ax.set_title(f"Pooled TCP (n={len(modeling)})")
ax.legend(frameon=False)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "03_tcp_curves_os_proxy.png")
plt.close(fig)

comparison_df, _ = run_model_comparison(doses, outcomes, frame=modeling)
comparison_df[["model", "aic", "roc_auc", "hl_p_value"]]


,model,aic,roc_auc,hl_p_value
0,probit_tcp,241.731266,0.683455,0.100941
1,logistic_tcp,241.745486,0.683455,0.100755
2,poisson_tcp,241.818325,0.683455,0.100260
3,eud_tcp,243.690797,0.684951,0.784179


In [4]:
metrics_dir = REPORTS_DIR / "metrics"
rano_cmp, _, _ = run_rano_tcp_comparison(modeling, metrics_dir)
rano_cmp[["model", "endpoint", "roc_auc_insample", "roc_auc_cv_mean", "lr_p_value"]]


,model,endpoint,roc_auc_insample,roc_auc_cv_mean,lr_p_value
0,poisson_tcp,os_median_proxy,0.622172,0.621279,0.010232
1,poisson_tcp,rano_non_pd_t1,0.425744,0.422109,1.000000
2,logistic_tcp,os_median_proxy,0.622172,0.621279,0.010436
3,logistic_tcp,rano_non_pd_t1,0.425744,0.422109,1.000000
4,probit_tcp,os_median_proxy,0.622172,0.621279,0.010361
5,probit_tcp,rano_non_pd_t1,0.425744,0.422109,1.000000


In [5]:
within_df, cox_rano = run_within_arm_rano_analysis(modeling, metrics_dir)
within_df[["scheme", "metric", "n_rano", "spearman_rho", "spearman_p", "poisson_auc", "poisson_lr_p"]]


,scheme,metric,n_rano,spearman_rho,spearman_p,poisson_auc,poisson_lr_p
0,60Gy_30fr,Dmean_gy,96,-0.040602,0.694490,0.473626,1.000000
1,60Gy_30fr,D95_gy,96,0.067670,0.512406,0.543956,0.973132
2,60Gy_30fr,volume_cc,96,0.239383,0.018824,0.655495,0.099530
3,60Gy_30fr,gEUD_a10_gy,96,-0.037219,0.718837,0.475824,1.000000
4,60Gy_30fr,HI_gy,96,-0.025376,0.806132,NaN,NaN
5,40Gy_15fr,Dmean_gy,34,0.012697,0.943183,0.510345,0.970516
6,40Gy_15fr,D95_gy,34,-0.190458,0.280622,0.344828,1.000000
7,40Gy_15fr,volume_cc,34,0.410542,0.015878,0.834483,0.037224
8,40Gy_15fr,gEUD_a10_gy,34,0.071951,0.685939,0.558621,0.953133
9,40Gy_15fr,HI_gy,34,0.300500,0.084199,NaN,NaN


In [6]:
cox_rano[["term", "hazard_ratio", "p_value"]]


,term,hazard_ratio,p_value
0,age,1.012613,0.224379
1,sex_M,1.014366,0.936499
2,who_status,1.574357,0.000452
3,eqd2_gy,0.965376,0.009283
4,rano_controlled_t1,0.484544,0.000867
